# 롱 컨텍스트 처리 - 실습 코드 3: KV Cache 압축: 중요 토큰 선택적 유지

- Tutorial ID: `expand-long-context`
- Tutorial: 롱 컨텍스트 처리
- Section ID: `expand-long-context-code-3`
- Section: 실습 코드 3: KV Cache 압축: 중요 토큰 선택적 유지


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 3: KV Cache 압축: 중요 토큰 선택적 유지
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) autoregressive decoding에서 이전 K/V를 재사용해 계산을 줄이는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn

class KVCacheCompressor:
    """KV Cache 압축: 중요도 기반 토큰 선택적 유지
    
    H2O (Heavy-Hitter Oracle) 스타일 구현:
    - 최근 N개 토큰 + Accumulated Attention이 높은 토큰 유지
    - 나머지 토큰의 KV는 삭제하여 메모리 절약
    """
        def __init__(self, budget_ratio=0.5, recent_window=64):
        """
        budget_ratio: 유지할 KV의 비율 (0.5 = 50% 유지)
        recent_window: 항상 유지할 최근 토큰 수
        """
        self.budget_ratio = budget_ratio
        self.recent_window = recent_window
        self.accumulated_attn = {}  # layer_id → accumulated attention scores
    
        def compute_importance(self, attn_weights, layer_id):
        """Attention 가중치를 누적하여 토큰 중요도 계산"""
        # attn_weights: (batch, heads, seq_q, seq_k)
        # 각 키 토큰이 받은 attention의 합 = 중요도
        importance = attn_weights.sum(dim=(0, 1, 2))  # (seq_k,)
        
        if layer_id not in self.accumulated_attn:
            self.accumulated_attn[layer_id] = torch.zeros_like(importance)
        self.accumulated_attn[layer_id] += importance
        
        return self.accumulated_attn[layer_id]
    
        def select_tokens(self, importance_scores, seq_len):
        """유지할 토큰 인덱스 선택"""
        budget = int(seq_len * self.budget_ratio)
        
        # 최근 토큰은 항상 유지
        recent_indices = torch.arange(
            max(0, seq_len - self.recent_window), seq_len
        )
        
        # 나머지 예산에서 중요도 높은 토큰 선택
        remaining_budget = budget - len(recent_indices)
        if remaining_budget > 0:
            # 최근 윈도우 제외한 이전 토큰
            older_indices = torch.arange(0, max(0, seq_len - self.recent_window))
            if len(older_indices) > 0:
                older_importance = importance_scores[older_indices]
                top_k_indices = older_indices[torch.topk(older_importance, 
                    min(remaining_budget, len(older_indices))).indices]
                selected = torch.cat([top_k_indices, recent_indices]).sort()[0]
            else:
                selected = recent_indices
        else:
            selected = recent_indices
        
        return selected
    
        def compress_kv_cache(self, key_cache, value_cache, attn_weights, layer_id):
        """KV Cache 압축 실행
        
        key_cache: (batch, heads, seq, dim)
        value_cache: (batch, heads, seq, dim)
        """
                seq_len = key_cache.size(2)
        
        # 중요도 계산
        importance = self.compute_importance(attn_weights, layer_id)
        
        # 유지할 토큰 선택
        selected = self.select_tokens(importance, seq_len)
        
        # KV Cache 압축
        compressed_keys = key_cache[:, :, selected, :]
        compressed_values = value_cache[:, :, selected, :]
        
        compression_ratio = 1 - (len(selected) / seq_len)
        
        return compressed_keys, compressed_values, compression_ratio

# ── 시뮬레이션 ──
print("=== KV Cache 압축 시뮬레이션 ===\n")
batch, heads, seq_len, dim = 1, 8, 4096, 128
compressor = KVCacheCompressor(budget_ratio=0.3, recent_window=64)

# 더미 KV Cache
key_cache = torch.randn(batch, heads, seq_len, dim)
value_cache = torch.randn(batch, heads, seq_len, dim)
attn_weights = torch.rand(batch, heads, 1, seq_len)
attn_weights = attn_weights / attn_weights.sum(dim=-1, keepdim=True)

# 특정 위치에 높은 attention 부여 (중요 토큰 시뮬레이션)
attn_weights[0, :, :, [0, 1, 2, 100, 500, 1000]] *= 10  # Attention Sink + 중요 토큰
attn_weights = attn_weights / attn_weights.sum(dim=-1, keepdim=True)

original_mb = (key_cache.numel() + value_cache.numel()) * 2 / 1e6  # FP16
print(f"원래 KV Cache: {seq_len} 토큰 = {original_mb:.1f} MB")

for budget in [0.5, 0.3, 0.2, 0.1]:
    comp = KVCacheCompressor(budget_ratio=budget, recent_window=64)
    ck, cv, ratio = comp.compress_kv_cache(key_cache, value_cache, attn_weights, layer_id=0)
    new_mb = (ck.numel() + cv.numel()) * 2 / 1e6
    print(f"  예산 {budget*100:.0f}%: {ck.size(2)} 토큰 유지 = {new_mb:.1f} MB ({ratio*100:.0f}% 압축)")